# Üye C — BLIP / Kaggle
Bu notebook henüz GPU üzerinde çalıştırılmamıştır. Skor içermez.
Kaggle'da GPU seçin ve Internet'i açın. Add Input ile adityajn105/flickr8k veri setini ekleyin.
Güncel proje kodunu ZIP olarak ayrı bir özel Kaggle Dataset'e yükleyip notebook'a ekleyin.
ZIP içinde src/, app/, configs/ olmalı; veri, checkpoint, .git veya kimlik bilgileri bulunmamalı.
Kaggle Input yolları değişebilir; aşağıdaki ilk hücrede gerçek yolları kontrol edin.
Test kümesi bu notebook'ta kullanılmaz.


In [ ]:
from pathlib import Path
inputs = Path('/kaggle/input')
datasets = [p.parent for p in inputs.rglob('captions.txt') if (p.parent/'Images').is_dir()]
archives = list(inputs.rglob('image-captioning-code.zip'))
expanded = [p.parent.parent for p in inputs.rglob('blip.py') if p.parent.name == 'src']
assert len(datasets) == 1, f'Flickr8k Images ve captions.txt yollarını kontrol edin: {datasets}'
assert len(archives) + len(expanded) == 1, f'Tek proje kaynağı ekleyin: {archives + expanded}'
DATA = datasets[0]
PROJECT = (archives + expanded)[0]
print('Flickr8k:', DATA, 'Proje:', PROJECT)


In [ ]:
import os, shutil, zipfile
WORK = Path('/kaggle/working/image-captioning')
WORK.mkdir(exist_ok=True)
if PROJECT.is_dir():
    shutil.copytree(PROJECT, WORK, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(PROJECT) as archive:
        for name in archive.namelist():
            assert (WORK/name).resolve().is_relative_to(WORK.resolve()), 'Unsafe ZIP path'
        archive.extractall(WORK)
assert (WORK/'src/blip.py').is_file()
os.chdir(WORK)


In [ ]:
# Kaggle'ın CUDA uyumlu torch/torchvision kurulumunu koruyun.
%pip install "transformers==4.57.1" "pycocoevalcap==1.2" PyYAML tqdm Pillow
import torch
assert torch.cuda.is_available(), 'Notebook için GPU seçin'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True)
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_blip.py', '-v')
run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_evaluate.py', '-v')


## 1. Fine-tuning öncesi doğrulama
İki yöntem de aynı 1000 doğrulama görüntüsünü kullanır. Hugging Face ağırlıkları ilk çalıştırmada indirilir.


In [ ]:
run('-m','src.blip','pretrained','--images-dir',DATA/'Images','--captions',DATA/'captions.txt',
    '--output','results/model4_blip_pretrained')


## 2. Fine-tuning ve en iyi checkpoint'in doğrulaması
AdamW 5e-6, batch 8, en fazla 5 epoch; encoder ve decoder birlikte eğitilir.
AMP, pad=-100, gradient clipping=5. En iyi checkpoint doğrulama greedy CIDEr ile seçilir.
Kesilirse checkpoints/ altındaki best ve last HF klasörleri ağırlıkları korur;
optimizer durumundan resume bu ilk sürümde yoktur. Yeniden başlatırken yeni çıktı klasörleri seçin.
GPU bellek hatasında batch'i sessizce değiştirmeyin; hata çıktısını paylaşın.


In [ ]:
run('-m','src.blip','finetune','--images-dir',DATA/'Images','--captions',DATA/'captions.txt',
    '--output','results/model4_blip_finetuned','--checkpoint','checkpoints/model4_blip','--epochs','5')


In [ ]:
run('-m','src.results_table')
print((WORK/'results/val_table.md').read_text())


## Çıktıları saklama
Kaggle oturumu bitmeden Save Version ile çıktıları kaydedin.
results/model4_blip_pretrained ve results/model4_blip_finetuned klasörlerini indirin.
checkpoints/model4_blip/best klasörünü ayrıca saklayın: model.safetensors ile işlemci/tokenizer dosyaları birlikte gerekir.
Checkpoint Git'e yüklenmez. Demo config'indeki model_id bu klasöre yönlendirilir.
METEOR için tahmin dosyaları İbrahim'e verilebilir; mevcut komutlar METEOR çalıştırmaz.
environment.txt gerçekten kullanılan sürümleri kaydeder; ekip requirements.txt kilidi buna ve demo testine göre yapılmalıdır.
